# SmartRead Agent V0.4 句子重要性分析模型

## 项目说明

本 Notebook 用于课程项目中的端侧 AI 演示。模型采用人工设计特征和弱监督标签训练轻量模型，重点体现移动端模型训练、转换和部署流程，不使用真实大规模语料，也不声称模型具备商业级准确率。

## 模型目标

模型输入 5 维句子特征，输出 0 到 1 的句子重要性分数。Android 端将分数映射为高/中/低，并在摘要结果页展示 LiteRT 本地模型推理来源。

## 特征设计

每句话输入 5 个数值特征：

1. `sentenceLengthNorm`：句子长度归一化。
2. `keywordOverlapScore`：句子与关键词的重合度。
3. `positionScore`：句子在文章中的位置得分，靠前句子略高。
4. `punctuationScore`：是否包含冒号、分号等提示性标点。
5. `summaryCueScore`：是否包含“因此、总之、主要、核心、说明、体现、可以看出”等提示词。

模型输出 0 到 1 的重要性分数，Android 端按阈值转换为高/中/低。

In [ ]:
import pathlib
import numpy as np

PROJECT_ROOT = pathlib.Path.cwd().parents[1] if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
MODEL_DIR = PROJECT_ROOT / 'model'
EXPORT_DIR = MODEL_DIR / 'exports'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR

## 构造弱监督训练数据

训练数据不是人工标注数据集，而是根据特征权重规则生成的弱监督样本。这样适合课程演示：可以说明训练、转换、部署流程，同时避免伪造真实数据来源。

In [ ]:
def make_weak_supervised_data(seed=42):
    rng = np.random.default_rng(seed)
    rows, labels = [], []
    for _ in range(960):
        length = rng.beta(2.2, 3.0)
        keyword_overlap = rng.beta(1.6, 4.2)
        position = rng.beta(2.0, 2.8)
        punctuation = rng.choice([0.0, 0.35, 0.7, 1.0], p=[0.52, 0.24, 0.18, 0.06])
        cue = rng.choice([0.0, 0.45, 0.8, 1.0], p=[0.58, 0.22, 0.16, 0.04])
        score = 0.22*length + 0.31*keyword_overlap + 0.20*position + 0.12*punctuation + 0.25*cue - 0.05
        score += rng.normal(0.0, 0.035)
        rows.append([length, keyword_overlap, position, punctuation, cue])
        labels.append([np.clip(score, 0.02, 0.98)])
    return np.asarray(rows, dtype=np.float32), np.asarray(labels, dtype=np.float32)

x_train, y_train = make_weak_supervised_data()
x_train.shape, y_train.shape, y_train[:5].ravel()

## 轻量模型结构

计划模型结构为 `Input(5) -> Dense(8, relu) -> Dense(4, relu) -> Dense(1, sigmoid)`。该结构参数量很小，适合课程演示中的移动端本地推理。

## 训练与评估

如果环境安装了 TensorFlow，可以使用下面的 Keras Sequential 模型训练。该 Notebook 用于记录模型训练与转换流程，实际运行结果以本地环境为准。

## 导出 TFLite / LiteRT 模型

如果环境安装了 TensorFlow，可以使用下面的 Keras Sequential 模型训练并转换为 `.tflite`。当前 Codex 本机 Python 环境没有 TensorFlow，且网络代理阻止安装，因此本项目实际执行 `model/train_sentence_importance_model.py`，用 NumPy 训练同结构网络，并通过 TensorFlow Lite FlatBuffer schema 生成真实 `.tflite` 文件。

In [ ]:
# TensorFlow 可用时运行本单元。
try:
    import tensorflow as tf
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(5,)),
        tf.keras.layers.Dense(8, activation='relu'),
        tf.keras.layers.Dense(4, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid'),
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    history = model.fit(x_train, y_train, epochs=40, batch_size=32, validation_split=0.2, verbose=1)
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_model = converter.convert()
    (EXPORT_DIR / 'sentence_importance_model.tflite').write_bytes(tflite_model)
    print('已导出 TensorFlow Lite 模型')
except Exception as exc:
    print('当前环境未运行 Keras 路径：', type(exc).__name__, exc)

## 当前项目实际执行脚本

在本机环境中执行以下脚本，已经生成：

- `model/exports/sentence_importance_model.tflite`
- `model/exports/labels.txt`
- `model/exports/sentence_importance_training_summary.json`
- `model/exports/model_metadata.json`

In [ ]:
# 在项目根目录运行：
# python model/train_sentence_importance_model.py

import json
summary_path = EXPORT_DIR / 'sentence_importance_training_summary.json'
if summary_path.exists():
    print(summary_path.read_text(encoding='utf-8'))
else:
    print('尚未生成训练摘要，请先运行脚本。')

## Android 集成说明

Android 工程将 `sentence_importance_model.tflite` 放入 `app/src/main/assets/`，通过 `SentenceImportanceClassifier` 使用 TFLite Interpreter 加载模型。摘要结果页会显示“LiteRT 端侧分析”卡片和“LiteRT 本地模型”来源。

## 简单推理验证

Android 端最终使用 TFLite Interpreter 加载模型并执行推理。Notebook 中的推理验证可参考训练摘要中的 `sample_score`。

## 局限性说明

当前模型使用弱监督样本，适合展示训练、转换、部署、推理流程，不代表真实人工标注语料上的最终效果。后续可以补充真实阅读样本和人工重要性标注。

## 实验总结

V0.4 模型使用轻量数值特征和小型神经网络，目标是为 Android App 增加端侧 AI 技术亮点。当前模型输入小、推理成本低，适合在移动端演示句子重要性评分。模型不足是训练标签来自弱监督规则，不代表真实人工标注质量，后续可收集真实样本或接入 LiteRT 更完整的模型评估流程。